# Week 7: Deep Learning Foundations
## Notebook 15: Deep Residual Connections & Identity Mappings

### 1. Conceptual Intuition & Theoretical Background
Before ResNets (He et al., 2015), stacking more layers caused training degradation: a 56-layer network had higher training error than a 20-layer network, despite having strictly higher representational capacity.

**Residual Learning Formulation:**
Instead of fitting an underlying mapping $\mathcal{H}(x)$, the network fits the residual $\mathcal{F}(x) = \mathcal{H}(x) - x$:
$$y = \mathcal{F}(x, W) + x$$

**The Identity Highway Gradient Proof:**
Applying the chain rule across $L$ residual blocks:
$$x_L = x_l + \sum_{i=l}^{L-1} \mathcal{F}(x_i, W_i)$$
Taking partial derivatives with respect to earlier layer activation $x_l$:
$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \frac{\partial x_L}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \left( 1 + \frac{\partial}{\partial x_l} \sum_{i=l}^{L-1} \mathcal{F}(x_i, W_i) \right)$$

The $+1$ term forms an unobstructed **gradient highway**: even if the residual gradients $\frac{\partial \mathcal{F}}{\partial x_l}$ vanish, the gradient $\frac{\partial \mathcal{L}}{\partial x_L}$ flows backward across layers unchanged!

---

### 2. Scratch Python Implementation: ResNet Identity Highway

In [ ]:
import numpy as np

# Simulation: Backpropagation through 30 layers
n_layers = 30
dim = 64
np.random.seed(42)

# Residual Block: y = relu(W @ x) + x
# Plain Block:    y = relu(W @ x)

def simulate_backprop(use_residual=True):
    # Simulate weights initialized with slight shrinkage
    weights = [np.random.normal(0, 0.08, (dim, dim)) for _ in range(n_layers)]
    
    # Forward pass
    x = np.random.randn(dim)
    activations = [x]
    for w in weights:
        res = np.dot(w, activations[-1])
        if use_residual:
            out = res + activations[-1] # Identity skip connection
        else:
            out = res
        activations.append(out)
        
    # Backward pass: track gradient magnitude flowing back to layer 0
    grad = np.random.randn(dim) # Output gradient
    grad_norms = [np.linalg.norm(grad)]
    
    for i in reversed(range(n_layers)):
        w = weights[i]
        # Jacobian approximation
        df = w
        if use_residual:
            # Gradient highway: (df + I)
            grad = np.dot(df.T, grad) + grad
        else:
            grad = np.dot(df.T, grad)
        grad_norms.append(np.linalg.norm(grad))
        
    return activations, grad_norms

_, grads_plain = simulate_backprop(use_residual=False)
_, grads_res = simulate_backprop(use_residual=True)

print("--- Residual Connection Gradient Highway ---")
print(f"Top Layer Output Gradient Norm:      {grads_res[0]:.2f}")
print(f"Plain Network Layer 0 Gradient Norm: {grads_plain[-1]:.6e} (Vanished completely to 0!)")
print(f"ResNet Layer 0 Gradient Norm:        {grads_res[-1]:.2f} (Gradient preserved across 30 layers!)")

### 3. Step-by-Step Code Walkthrough
1. **Additive Identity:** The $+x$ skip connection creates an additive pathway for activations.
2. **Gradient Highway:** In backpropagation, the identity Jacobian adds $I$, guaranteeing non-vanishing gradient flow.
3. **Degradation Resolution:** Allows deep networks ($>100$ layers) to optimize efficiently.

---

### 4. Industry Applications & Trade-offs
- **Transformers & Modern LLMs:** Every self-attention and MLP block uses residual connections ($x + \text{Sublayer}(x)$).
- **Trade-off:** Pre-LN vs Post-LN positioning affects training stability at massive scales.

---

### 5. Top Interview Questions & Answers
**Q1: Why is the degradation problem in deep networks not caused by vanishing gradients?**
> **Answer:** With proper initialization and normalization, gradients remain non-zero. The degradation problem occurs because optimizing deep non-linear functions is hard: deep networks struggle to learn simple identity mappings without residual connections.

**Q2: What is the difference between Pre-LN and Post-LN in residual networks?**
> **Answer:** Post-LN applies normalization after the residual addition ($x_{t+1} = \text{LN}(x_t + F(x_t))$), which can cause gradient instability in early layers. Pre-LN normalizes the sublayer input before addition ($x_{t+1} = x_t + F(\text{LN}(x_t))$), providing a cleaner identity highway.